<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Space X  Falcon 9 First Stage Landing Prediction**


## Web scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia


Estimated time needed: **40** minutes


In this lab, you will be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/Falcon9_rocket_family.svg)


Falcon 9 first stage will land successfully


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/crash.gif)


More specifically, the launch records are stored in a HTML table shown below:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/falcon9-launches-wiki.png)


  ## Objectives
Web scrap Falcon 9 launch records with `BeautifulSoup`: 
- Extract a Falcon 9 launch records HTML table from Wikipedia
- Parse the table and convert it into a Pandas data frame


First let's import required packages for this lab


In [9]:
import requests
import unicodedata
from bs4 import BeautifulSoup


In [10]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(static_url, headers=headers)
page = response.text


In [7]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(page, "html.parser")


In [8]:
html_tables = soup.find_all('table')
print("Number of tables found:", len(html_tables))


Number of tables found: 25


and we will provide some helper functions for you to process web scraped HTML table


In [11]:
import unicodedata

def date_time(table_cells):
    """
    Returns the separated date and time from the HTML table cell.
    Input: element of a table data cell
    Output: [date, time]
    """
    data_time = list(table_cells.strings)[:2]
    return [data_time[0].strip(), data_time[1].strip()]

def booster_version(table_cells):
    """
    Extracts booster version from the HTML table cell.
    Input: element of a table data cell
    Output: string of booster version
    """
    out = ' '.join([s for s in table_cells.strings if s.strip() != ''])
    return out.strip()

def landing_status(table_cells):
    """
    Gets landing status from the HTML table cell.
    Input: element of a table data cell
    Output: string of landing status
    """
    out = [s for s in table_cells.strings]
    return out[0].strip() if out else ""

def get_mass(table_cells):
    """
    Extracts payload mass as a string from the cell (removes footnotes, finds 'kg').
    Input: table cell element
    Output: mass value in kg as a string
    """
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        idx = mass.find("kg")
        new_mass = mass[:idx+2] if idx != -1 else "0"
    else:
        new_mass = "0"
    return new_mass

def extract_column_from_header(row):
    """
    Returns cleaned column name from the HTML table header row.
    Input: header row element
    Output: header as string
    """
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    column_name = ' '.join(row.contents)
    if not column_name.strip().isdigit():
        column_name = column_name.strip()
    return column_name


In [12]:
print(page[:1000])


<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 vector-feature-night-mode-enabled skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available" lang="en" dir="ltr">
<head>
<meta charset="UTF-8">
<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>
<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width

To keep the lab tasks consistent, you will be asked to scrape the data from a snapshot of the  `List of Falcon 9 and Falcon Heavy launches` Wikipage updated on
`9th June 2021`


In [ ]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

Next, request the HTML page from the above URL and get a `response` object


### TASK 1: Request the Falcon9 Launch Wiki page from its URL


First, let's perform an HTTP GET method to request the Falcon9 Launch HTML page, as an HTTP response.


In [ ]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(static_url, headers=headers)
page = response.text


Create a `BeautifulSoup` object from the HTML `response`


In [ ]:
# Use BeautifulSoup() to create a BeautifulSoup object from a response text content
soup = BeautifulSoup(page, "html.parser")


Print the page title to verify if the `BeautifulSoup` object was created properly 


In [ ]:
# Use soup.title attribute
soup.title


### TASK 2: Extract all column/variable names from the HTML table header


Next, we want to collect all relevant column names from the HTML table header


Let's try to find all tables on the wiki page first. If you need to refresh your memory about `BeautifulSoup`, please check the external reference link towards the end of this lab


In [ ]:
def extract_column_from_header(row):
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    column_name = ' '.join(row.contents)
    if not (column_name.strip().isdigit()):
        column_name = column_name.strip()
    return column_name


Starting from the third table is our target table contains the actual launch records.


In [ ]:
# Show how many tables were found on the page
print(f"Number of tables found: {len(html_tables)}")


In [ ]:
# Let's print the third table and check its content
first_launch_table = html_tables[2]
print(first_launch_table)

In [ ]:
# Print the content of each table to help find the correct index
for idx, table in enumerate(html_tables):
    print(f"Table {idx}:")
    print(table)
    print("-"*40)


You should able to see the columns names embedded in the table header elements `<th>` as follows:


In [ ]:
# See how many tables are found — just copy/paste and run!
print(f"Number of tables found: {len(html_tables)}")


```
<tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a> <sup class="reference" id="cite_ref-booster_11-0"><a href="#cite_note-booster-11">[b]</a></sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference" id="cite_ref-Dragon_12-0"><a href="#cite_note-Dragon-12">[c]</a></sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 9 first-stage landing tests">Booster<br/>landing</a>
</th></tr>
```


In [ ]:
# Print every table — lets you visually spot the Falcon 9 launch table
for idx, table in enumerate(html_tables):
    print(f"Table {idx}:")
    print(table)
    print("-"*40)


Next, we just need to iterate through the `<th>` elements and apply the provided `extract_column_from_header()` to extract column name one by one


In [ ]:
column_names = []

# Apply find_all() function with `th` element on first_launch_table
elements = first_launch_table.find_all('th')

# Iterate each th element and apply the provided extract_column_from_header() to get a column name
# Append the Non-empty column name (`if name is not None and len(name) > 0`) into a list called column_names
for th in elements:
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)


Check the extracted column names


In [ ]:
print(column_names)

## TASK 3: Create a data frame by parsing the launch HTML tables


We will create an empty dictionary with keys from the extracted column names in the previous task. Later, this dictionary will be converted into a Pandas dataframe


In [15]:
launch_dict = {
    'Flight No.': [],
    'Date': [],
    'Time': [],
    'Version Booster': [],
    'Launch site': [],
    'Payload': [],
    'Payload mass': [],
    'Orbit': [],
    'Customer': [],
    'Launch outcome': [],
    'Booster landing': []
}


In [ ]:
import unicodedata
import pandas as pd

def date_time(table_cells):
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    out = ''.join([booster_version for i, booster_version in enumerate(table_cells.strings) if i % 2 == 0])[0:-1]
    return out

def landing_status(table_cells):
    out = [i for i in table_cells.strings][0]
    return out

def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        idx = mass.find("kg")
        new_mass = mass[0:idx+2] if idx != -1 else "0"
    else:
        new_mass = "0"
    return new_mass

# Initialize the dictionary for your DataFrame
launch_dict = {
    'Flight No.': [],
    'Date': [],
    'Time': [],
    'Version Booster': [],
    'Launch site': [],
    'Payload': [],
    'Payload mass': [],
    'Orbit': [],
    'Customer': [],
    'Launch outcome': [],
    'Booster landing': []
}

# Parse only the table with the launches ("wikitable plainrowheaders collapsible")
for table in soup.find_all('table', "wikitable plainrowheaders collapsible"):
    for row in table.find_all("tr"):
        if row.th and row.th.string and row.th.string.strip().isdigit():
            flight_number = row.th.string.strip()
            cells = row.find_all('td')
            if len(cells) >= 9:
                datatimelist = date_time(cells[0])
                date = datatimelist[0].strip(',')
                time = datatimelist[1]
                bv = booster_version(cells[1])
                if not bv and cells[1].a:
                    bv = cells[1].a.string
                launch_site = cells[2].a.string if cells[2].a else None
                payload = cells[3].a.string if cells[3].a else None
                payload_mass = get_mass(cells[4])
                orbit = cells[5].a.string if cells[5].a else None
                customer = cells[6].a.string if cells[6].a else None
                launch_outcome = list(cells[7].strings)[0].strip()
                booster_landing = landing_status(cells[8])

                launch_dict['Flight No.'].append(flight_number)
                launch_dict['Date'].append(date)
                launch_dict['Time'].append(time)
                launch_dict['Version Booster'].append(bv)
                launch_dict['Launch site'].append(launch_site)
                launch_dict['Payload'].append(payload)
                launch_dict['Payload mass'].append(payload_mass)
                launch_dict['Orbit'].append(orbit)
                launch_dict['Customer'].append(customer)
                launch_dict['Launch outcome'].append(launch_outcome)
                launch_dict['Booster landing'].append(booster_landing)

df = pd.DataFrame(launch_dict)
print(df.head())


In [16]:
# This assumes you have already run the soup and html_tables cells,
# and have defined/initialized launch_dict keys with empty lists.

# Example: launch_dict = { 'Flight No.': [], 'Date': [], ... }

first_launch_table = html_tables[2]  # adjust index as needed if your table is different

for row in first_launch_table.find_all('tr'):
    # Skip header rows
    if row.th and row.th.string and row.th.string.strip().isdigit():
        flight_number = row.th.string.strip()
        cells = row.find_all('td')
        
        if len(cells) < 9:   # Not enough cells, skip (defensive)
            continue

        date, time = date_time(cells[0])
        bv = booster_version(cells[1])
        launch_site = cells[2].a.string if cells[2].a else None
        payload = cells[3].a.string if cells[3].a else None
        payload_mass = get_mass(cells[4])
        orbit = cells[5].a.string if cells[5].a else None
        customer = cells[6].a.string if cells[6].a else None
        launch_outcome = list(cells[7].strings)[0].strip()
        booster_landing = landing_status(cells[8])

        # Append to dictionary
        launch_dict['Flight No.'].append(flight_number)
        launch_dict['Date'].append(date)
        launch_dict['Time'].append(time)
        launch_dict['Version Booster'].append(bv)
        launch_dict['Launch site'].append(launch_site)
        launch_dict['Payload'].append(payload)
        launch_dict['Payload mass'].append(payload_mass)
        launch_dict['Orbit'].append(orbit)
        launch_dict['Customer'].append(customer)
        launch_dict['Launch outcome'].append(launch_outcome)
        launch_dict['Booster landing'].append(booster_landing)


Next, we just need to fill up the `launch_dict` with launch records extracted from table rows.


Usually, HTML tables in Wiki pages are likely to contain unexpected annotations and other types of noises, such as reference links `B0004.1[8]`, missing values `N/A [e]`, inconsistent formatting, etc.


To simplify the parsing process, we have provided an incomplete code snippet below to help you to fill up the `launch_dict`. Please complete the following code snippet with TODOs or you can choose to write your own logic to parse all launch tables:


In [ ]:
import unicodedata
import pandas as pd

def date_time(table_cells):
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    out = ''.join([booster_version for i, booster_version in enumerate(table_cells.strings) if i % 2 == 0])[0:-1]
    return out

def landing_status(table_cells):
    out = [i for i in table_cells.strings][0]
    return out

def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        idx = mass.find("kg")
        new_mass = mass[0:idx+2] if idx != -1 else "0"
    else:
        new_mass = "0"
    return new_mass

# Initialize the dictionary for your DataFrame
launch_dict = {
    'Flight No.': [],
    'Date': [],
    'Time': [],
    'Version Booster': [],
    'Launch site': [],
    'Payload': [],
    'Payload mass': [],
    'Orbit': [],
    'Customer': [],
    'Launch outcome': [],
    'Booster landing': []
}

for row in first_launch_table.find_all("tr"):
    if row.th and row.th.string and row.th.string.strip().isdigit():
        flight_number = row.th.string.strip()
        cells = row.find_all('td')
        if len(cells) >= 9:
            # Date and Time
            datatimelist = date_time(cells[0])
            date = datatimelist[0].strip(',')
            time = datatimelist[1]
            # Booster version
            bv = booster_version(cells[1])
            if not bv and cells[1].a:
                bv = cells[1].a.string
            # Launch site
            launch_site = cells[2].a.string if cells[2].a else None
            # Payload
            payload = cells[3].a.string if cells[3].a else None
            # Payload mass
            payload_mass = get_mass(cells[4])
            # Orbit
            orbit = cells[5].a.string if cells[5].a else None
            # Customer
            customer = cells[6].a.string if cells[6].a else None
            # Launch outcome
            launch_outcome = list(cells[7].strings)[0].strip()
            # Booster landing
            booster_landing = landing_status(cells[8])
            
            # Append to dictionary
            launch_dict['Flight No.'].append(flight_number)
            launch_dict['Date'].append(date)
            launch_dict['Time'].append(time)
            launch_dict['Version Booster'].append(bv)
            launch_dict['Launch site'].append(launch_site)
            launch_dict['Payload'].append(payload)
            launch_dict['Payload mass'].append(payload_mass)
            launch_dict['Orbit'].append(orbit)
            launch_dict['Customer'].append(customer)
            launch_dict['Launch outcome'].append(launch_outcome)
            launch_dict['Booster landing'].append(booster_landing)

# Create DataFrame
df = pd.DataFrame(launch_dict)
print(df.head())


In [ ]:
extracted_row = 0

for table_number, table in enumerate(soup.find_all('table', "wikitable plainrowheaders collapsible")):
    for rows in table.find_all("tr"):
        if rows.th:
            if rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
        else:
            flag = False

        row = rows.find_all('td')

        if flag:
            extracted_row += 1

            # Flight Number
            launch_dict['Flight No.'].append(flight_number)

            # Date and Time
            datatimelist = date_time(row[0])
            date = datatimelist[0].strip(',')
            time = datatimelist[1]
            launch_dict['Date'].append(date)
            launch_dict['Time'].append(time)

            # Booster Version
            bv = booster_version(row[1])
            if not bv and row[1].a:
                bv = row[1].a.string
            launch_dict['Version Booster'].append(bv)

            # Launch Site
            launch_site = row[2].a.string if row[2].a else None
            launch_dict['Launch site'].append(launch_site)

            # Payload
            payload = row[3].a.string if row[3].a else None
            launch_dict['Payload'].append(payload)

            # Payload Mass
            payload_mass = get_mass(row[4])
            launch_dict['Payload mass'].append(payload_mass)

            # Orbit
            orbit = row[5].a.string if row[5].a else None
            launch_dict['Orbit'].append(orbit)

            # Customer
            customer = row[6].a.string if row[6].a else None
            launch_dict['Customer'].append(customer)

            # Launch Outcome
            launch_outcome = list(row[7].strings)[0].strip()
            launch_dict['Launch outcome'].append(launch_outcome)

            # Booster Landing
            booster_landing = landing_status(row[8])
            launch_dict['Booster landing'].append(booster_landing)


After you have fill in the parsed launch record values into `launch_dict`, you can create a dataframe from it.


In [ ]:
df= pd.DataFrame({ key:pd.Series(value) for key, value in launch_dict.items() })

We can now export it to a <b>CSV</b> for the next section, but to make the answers consistent and in case you have difficulties finishing this lab. 

Following labs will be using a provided dataset to make each lab independent. 


<code>df.to_csv('spacex_web_scraped.csv', index=False)</code>


In [ ]:
df.to_csv('spacex_web_scraped.csv', index=False)

## Authors


<a href="https://www.linkedin.com/in/yan-luo-96288783/">Yan Luo</a>


<a href="https://www.linkedin.com/in/nayefaboutayoun/">Nayef Abou Tayoun</a>


<!--
## Change Log
-->


<!--
| Date (YYYY-MM-DD) | Version | Changed By | Change Description      |
| ----------------- | ------- | ---------- | ----------------------- |
| 2021-06-09        | 1.0     | Yan Luo    | Tasks updates           |
| 2020-11-10        | 1.0     | Nayef      | Created the initial version |
-->


Copyright © 2021 IBM Corporation. All rights reserved.
